In [12]:
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import importlib
import book
import functions1 
import config 

importlib.reload(book)
importlib.reload(functions1)
importlib.reload(config)

print(config.PROJECT_ROOT)
z = os.path.abspath(os.path.join(os.getcwd(), '..'))
print(z)


/Users/alexwebb/laptop_coding/risk_matrix
/Users/alexwebb/laptop_coding/risk_matrix


In [9]:
pf = book.IBKR_live_adj
fetch_csv_robust = functions1.fetch_csv_robust
params = config.params
sort_cols = functions1.sort_cols

GBPUSD = fetch_csv_robust('GBPUSD.FOREX', params=params)
GBPUSD = sort_cols(GBPUSD)
GBPUSD_last = GBPUSD.iloc[-1]
print('GBPUSD last:', GBPUSD_last)
USD_total = 0
GBP_total = 0
for h in pf:
    exp_usd = h.get("USD_exposure")
    exp_gbp = h.get("GBP_exposure")
    if (exp_usd is None and exp_gbp is None) or h.get('risk_fx') is False:
        print('...skipping', h['name'])
        continue
    print(h['name'],'usd', exp_usd, 'gbp', exp_gbp    )

    # print(h.get("name"),'GBP exp:' ,h.get("GBP_exposure"))
    # get gbpchf last price
    ticker   = h.get("ticker")
    px_df = fetch_csv_robust(f'{ticker}', params=params,)
    px = sort_cols(px_df).iloc[-1]
    # print('last:' ,px)
    position = h.get("position")
    gpx = .01 if h.get('gbx', True) else 1.0
    mkt_val = position * px * gpx
    print('currency:', h.get('ccy'),'mkt val: £',round(mkt_val))
    if exp_gbp:
        gbp_to_hedge = mkt_val * exp_gbp
        print('gbp to hedge: £',round(gbp_to_hedge))
        GBP_total += gbp_to_hedge

    if exp_usd:
        usd_to_hedge = mkt_val * exp_usd
        if h.get('ccy') == 'GBP':
            print('converting to usd')
            usd_to_hedge *= GBPUSD_last
        USD_total += usd_to_hedge
        print('usd to hedge: $',round(usd_to_hedge))

    print('')
print ('total: £',round(GBP_total))
print ('total: $',round(USD_total))


GBPUSD last: 1.3447
...skipping EMIM
HEAL usd 0.65 gbp 0.03
currency: GBP mkt val: £ 1536
gbp to hedge: £ 46
converting to usd
usd to hedge: $ 1343

IBM usd 1.0 gbp None
currency: USD mkt val: £ 2605
usd to hedge: $ 2605

SGLN usd 1.0 gbp None
currency: GBP mkt val: £ 2392
converting to usd
usd to hedge: $ 3216

VUAG usd 1.0 gbp None
currency: GBP mkt val: £ 2684
converting to usd
usd to hedge: $ 3609

WSML usd 0.58 gbp 0.04
currency: USD mkt val: £ 4269
gbp to hedge: £ 171
usd to hedge: $ 2476

...skipping SIKA
YCA usd 1.0 gbp None
currency: GBP mkt val: £ 1491
converting to usd
usd to hedge: $ 2005

VEU usd None gbp 0.13
currency: USD mkt val: £ 4952
gbp to hedge: £ 644

...skipping IWDC
...skipping CASH_CHF
...skipping CASH_GBP
...skipping CASH_USD
total: £ 861
total: $ 15255


{'computershare': [{'name': 'Unilever', 'ticker': 'ULVR.LON', 'ccy': 'GBP', 'gbx': True, 'value_chf': 25000}, {'name': 'Shell', 'ticker': 'SHEL.LON', 'ccy': 'GBP', 'gbx': True, 'value_chf': 13000}, {'name': 'NatWest', 'ticker': 'NWG.LON', 'ccy': 'GBP', 'gbx': True, 'value_chf': 5000}, {'name': 'Barclays', 'ticker': 'BARC.LON', 'ccy': 'GBP', 'gbx': True, 'value_chf': 5000}, {'name': 'Tesco', 'ticker': 'TSCO.LON', 'ccy': 'GBP', 'gbx': True, 'value_chf': 5000}, {'name': 'SWDA', 'ticker': 'SWDA.LON', 'ccy': 'GBP', 'gbx': True, 'value_chf': 12000}, {'name': 'EMIM', 'ticker': 'EMIM.LON', 'ccy': 'GBP', 'gbx': True, 'value_chf': 8000}, {'name': 'IBM', 'ticker': 'IBM', 'ccy': 'USD', 'gbx': False, 'value_chf': 4000}, {'name': 'ERNS', 'ticker': 'ERNS.LON', 'ccy': 'GBP', 'gbx': True, 'value_chf': 5000}]}
[{'name': 'EMIM', 'ticker': 'EMIM.LSE', 'ccy': 'GBP', 'gbx': True, 'position': 200}, {'name': 'ERNS', 'ticker': 'ERNS.LSE', 'ccy': 'GBP', 'GBP_exposure': 1.0, 'gbx': False, 'position': 89, 'risk_f